# Кандидатогенерация для поиска услуг

BM25, E5-small и обучаемое расширение запросов; объединение через RRF.

In [1]:
from collections import Counter, defaultdict
import gc
import hashlib
import json
import os
from pathlib import Path
import re

import bm25s
import numpy as np
import pandas as pd
from scipy import sparse
import snowballstemmer
import torch
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd()
DATA = ROOT / 'datasets'
MODEL_CACHE = ROOT / 'models' / 'huggingface'
ALLOW_MODEL_DOWNLOAD = os.getenv('AVITO_ALLOW_MODEL_DOWNLOAD', '0') == '1'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
DEPTH = 500
MODEL_ID = 'intfloat/multilingual-e5-small'
MODEL_REVISION = '614241f622f53c4eeff9890bdc4f31cfecc418b3'
QUERY_COLUMNS = ['search_query', 'search_location_id', 'search_is_delivery_search',
                 'search_infm_params_text', 'search_category']
ITEM_COLUMNS = ['item_id', 'item_title_raw', 'item_description_raw', 'item_infm_params_text',
                'item_rating', 'item_location_id', 'item_latitude', 'item_longitude']
FIELDS = {'item_title_raw': 3., 'item_infm_params_text': 2., 'item_description_raw': 1.}
STEMMER = snowballstemmer.stemmer('russian')
STOPWORDS = {'и', 'в', 'на', 'по', 'с', 'от', 'до', 'для'}
RATING_PATTERN = re.compile(r'рейтинг\s+пользователя\s+(\d+(?:[.,]\d+)?)\s+звезд\w*\s+и\s+выше')
LABEL_PATTERN = re.compile(r'\b(?:вид услуги|тип услуги|место оказания услуг|рейтинг пользователя)\b')
MODEL_SPEC = {'version': 1, 'min_query_df': 3, 'min_document_df': 3,
              'min_support': 3, 'max_document_df_share': .15,
              'params_weight': .25, 'support_shrinkage': 5., 'terms_per_feature': 12}
IGNORED_TERMS = STOPWORDS | {'услуг', 'услуга', 'услуги', 'вид', 'тип', 'место',
    'оказание', 'оказания', 'рейтинг', 'пользователя', 'звезды', 'выше', 'или'}
IGNORED_STEMS = set(STEMMER.stemWords(sorted(IGNORED_TERMS)))


def json_save(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')

def file_hash(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:24]

def tokenize(text):
    return re.findall(r'[а-яa-z0-9]+', str(text or '').lower().replace('ё', 'е'))

def normalized(text):
    return ' '.join(tokenize(text))

def top(scores, k=DEPTH, positive=False):
    scores = np.asarray(scores)
    indices = np.flatnonzero(scores > 0) if positive else np.arange(len(scores))
    if len(indices) > k:
        threshold = np.partition(scores[indices], -k)[-k]
        better = indices[scores[indices] > threshold]
        ties = indices[scores[indices] == threshold]
        indices = np.concatenate([better, ties[:k-len(better)]])
    return indices[np.lexsort((indices, -scores[indices]))]

required = ['train.parquet', 'benchmark_items.parquet', 'benchmark_queries.parquet']
assert all((DATA/name).is_file() for name in required), 'Положите три Parquet-файла в datasets/'
INPUT_HASHES = {name: file_hash(DATA/name) for name in required}
# Отпечаток данных и настроек предотвращает загрузку устаревшего кэша.
CACHE = (ROOT/'artifacts'/'final')/fingerprint({'inputs':INPUT_HASHES, 'version':'production-1',
    'model':MODEL_REVISION, 'fields':FIELDS, 'expansion':MODEL_SPEC, 'device':DEVICE})
CACHE.mkdir(parents=True, exist_ok=True)

c:\Users\qa1ro\OneDrive\Рабочий стол\projects\avito-test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Данные и география

Расширение обучается на train-парах. Центры локаций — медианы координат
объявлений. Для географии сохраняется выборка принятой модели;
разбиения задаются явно в `GEOGRAPHY_SPLITS`.


In [2]:
corpus = pd.read_parquet(DATA/'benchmark_items.parquet', columns=ITEM_COLUMNS)
corpus = corpus.sort_values('item_id').reset_index(drop=True)  # Стабильный порядок и разрешение ничьих.
queries = pd.read_parquet(DATA/'benchmark_queries.parquet')
assert corpus.item_id.is_unique and queries.query_id.is_unique

# Этап, seed, число текстов, ограничение объявлениями benchmark.
# Это зафиксированная выборка оценки географии, а не запуск валидации.
GEOGRAPHY_SPLITS = (
    ('initial', 42, 2452, False),
    ('dev', 42, 7000, True),
    ('control', 20260928, 2452, False),
    ('holdout', 20260929, 2452, False),
)

def geography_metadata(corpus):
    """Метаданные с той же обучающей выборкой, на которой оценена география."""
    columns = ['item_id', 'item_location_id', 'item_latitude', 'item_longitude']
    train = pd.read_parquet(DATA/'train.parquet', columns=columns+['search_query'])
    train['text_group'] = train.search_query.map(normalized)
    excluded = set()
    benchmark_ids = set(corpus.item_id)
    for stage, seed, size, benchmark_only in GEOGRAPHY_SPLITS:
        eligible = train[~train.text_group.isin(excluded)]
        if benchmark_only:
            eligible = eligible[eligible.item_id.isin(benchmark_ids)]
        texts = sorted(eligible.text_group.unique())
        # Разные seed отражают отдельные этапы разбиения. Их замена меняет
        # выборку, центры локаций и итоговые кандидаты.
        rng = np.random.default_rng(seed)
        selected = rng.permutation(texts)[:size] if stage == 'dev' else rng.choice(texts, size, replace=False)
        excluded.update(selected)
    train = train[~train.text_group.isin(excluded)]
    return pd.concat([corpus[columns], train[columns]], ignore_index=True).drop_duplicates('item_id')


## Обучаемое расширение запросов

Слова и биграммы запросов связываются с термами выбранных объявлений.
Вес заголовка — 1, параметров — 0.25; каждый текст запроса имеет одинаковый
суммарный вес. Positive PMI и поддержка минимум тремя текстами ограничивают
случайные связи. Добавляются до четырёх термов, исходные слова сохраняются.

In [3]:
def stemmed_words(text):
    return [t for t in STEMMER.stemWords(tokenize(text))
            if len(t)>1 and t not in IGNORED_STEMS and not t.isdigit()]

def query_features(words):
    return sorted(set(words) | {words[i]+' '+words[i+1] for i in range(len(words)-1)})

class LexicalExpansion:
    """Учимся переводить слова запросов в термы заголовков/параметров, без item-приоров."""
    def __init__(self, model_dir):
        self.model_dir = model_dir

    def fit(self, pairs):
        manifest = {'inputs':INPUT_HASHES, 'spec':MODEL_SPEC}
        self.model_dir.mkdir(parents=True, exist_ok=True)
        metadata_path = self.model_dir/'model.json'
        if metadata_path.exists():
            metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
            assert metadata['manifest'] == manifest, 'Stale lexical model cache'
            self.features, self.terms = metadata['features'], metadata['terms']
            self.feature_lookup = {t:i for i,t in enumerate(self.features)}
            self.associations = sparse.load_npz(self.model_dir/'associations.npz')
            self.query_idf = np.asarray(metadata['query_idf'], np.float32)
            return self
        frame = pairs.copy()
        frame['text_group'] = frame.search_query.map(normalized)
        frame = frame.drop_duplicates(['text_group','item_id'])
        groups = sorted(frame.text_group.unique())
        group_lookup = {g:i for i,g in enumerate(groups)}
        qwords = [stemmed_words(g) for g in groups]
        qfeatures = [query_features(w) for w in qwords]
        qcounts = Counter(t for features in qfeatures for t in features)
        self.features = sorted(t for t,n in qcounts.items() if n>=MODEL_SPEC['min_query_df'])
        self.feature_lookup = {t:i for i,t in enumerate(self.features)}
        items = frame.drop_duplicates('item_id')
        title_tokens = [set(stemmed_words(t)) for t in items.item_title_raw.fillna('')]
        params_tokens = [set(stemmed_words(t)) for t in items.item_infm_params_text.fillna('')]
        dcounts = Counter(t for ts,ps in zip(title_tokens,params_tokens) for t in ts|ps)
        self.terms = sorted(t for t,n in dcounts.items()
            if n>=MODEL_SPEC['min_document_df'] and n/len(items)<=MODEL_SPEC['max_document_df_share'])
        term_lookup = {t:i for i,t in enumerate(self.terms)}
        item_lookup = {t:i for i,t in enumerate(items.item_id)}
        ii,jj,vv = [],[],[]
        for i,(title,params) in enumerate(zip(title_tokens,params_tokens)):
            weights = {t:1. for t in title if t in term_lookup}
            for t in params:
                if t in term_lookup:
                    weights[t] = max(weights.get(t,0.), MODEL_SPEC['params_weight'])
            denominator = max(sum(weights.values()), 1e-12)
            for t,v in weights.items():
                ii.append(i); jj.append(term_lookup[t]); vv.append(v/denominator)
        item_matrix = sparse.csr_matrix((np.asarray(vv,np.float32),(ii,jj)),shape=(len(items),len(self.terms)))
        del ii,jj,vv,title_tokens,params_tokens
        gi = frame.text_group.map(group_lookup).to_numpy(np.int32)
        di = frame.item_id.map(item_lookup).to_numpy(np.int32)
        group_sizes = np.bincount(gi, minlength=len(groups))
        # Каждому тексту запроса даём одинаковый суммарный вес, независимо от числа кликов.
        interactions = sparse.csr_matrix((1/group_sizes[gi],(gi,di)),shape=(len(groups),len(items)),dtype=np.float32)
        document_matrix = (interactions @ item_matrix).tocsr()
        qi,qj = [],[]
        for i,features in enumerate(qfeatures):
            for t in features:
                if t in self.feature_lookup:
                    qi.append(i); qj.append(self.feature_lookup[t])
        query_matrix = sparse.csr_matrix((np.ones(len(qi),np.float32),(qi,qj)),shape=(len(groups),len(self.features)))
        del frame,items,qfeatures,qwords,qi,qj,interactions,item_matrix
        gc.collect()
        support_matrix = document_matrix.copy()
        support_matrix.data[:] = 1.
        support = (query_matrix.T @ support_matrix).tocsr()
        mass = (query_matrix.T @ document_matrix).tocsr()
        mass.sort_indices(); support.sort_indices()
        assert np.array_equal(mass.indices,support.indices) and np.array_equal(mass.indptr,support.indptr)
        qdf = np.asarray(query_matrix.sum(axis=0)).ravel()
        background = np.asarray(document_matrix.mean(axis=0)).ravel()
        self.query_idf = np.log1p(len(groups)/qdf).astype(np.float32)
        counts = np.repeat(qdf,np.diff(mass.indptr))
        conditional = mass.data/counts
        # Positive PMI выделяет специфичные связи; support подавляет случайные совпадения.
        lift = np.log(np.maximum(conditional/np.maximum(background[mass.indices],1e-12),1.))
        score = conditional*lift*support.data/(support.data+MODEL_SPEC['support_shrinkage'])
        score[support.data<MODEL_SPEC['min_support']] = 0
        mass.data = score.astype(np.float32)
        del query_matrix,document_matrix,support_matrix,support,counts,conditional,lift,score
        mass.eliminate_zeros()
        oi,oj,ov = [],[],[]
        for i in range(mass.shape[0]):
            start,end = mass.indptr[i:i+2]
            positions = top(mass.data[start:end],MODEL_SPEC['terms_per_feature'],positive=True)
            for j in positions:
                oi.append(i); oj.append(int(mass.indices[start+j])); ov.append(float(mass.data[start+j]))
        self.associations = sparse.csr_matrix((np.asarray(ov,np.float32),(oi,oj)),shape=mass.shape)
        sparse.save_npz(self.model_dir/'associations.npz',self.associations)
        json_save(metadata_path, {'manifest':manifest,'features':self.features,'terms':self.terms,
            'query_idf':self.query_idf.tolist()})
        return self

    def expand(self,text,limit=4):
        words = stemmed_words(text)
        original = set(words)
        values = defaultdict(float)
        for feature in query_features(words):
            i = self.feature_lookup.get(feature)
            if i is None:
                continue
            start,end = self.associations.indptr[i:i+2]
            # Биграмма задаёт контекст; её вес выше веса отдельного слова.
            weight = self.query_idf[i]*(2 if ' ' in feature else 1)
            for j,v in zip(self.associations.indices[start:end],self.associations.data[start:end]):
                term = self.terms[j]
                if term not in original:
                    values[term] += weight*float(v)
        ordered = sorted(values.items(),key=lambda x:(-x[1],x[0]))[:limit]
        if not ordered:
            return []
        maximum = ordered[0][1]
        return [(term,float(value/maximum)) for term,value in ordered]

pairs = pd.read_parquet(DATA/'train.parquet', columns=['search_query', 'item_id',
    'item_title_raw', 'item_infm_params_text'])
expansion = LexicalExpansion(CACHE/'query_expansion').fit(pairs)
del pairs
_ = gc.collect()

## E5-small

Префиксы `query:` / `passage:`. В объявление входят заголовок, до 64 слов
параметров и 160 слов описания; max_seq_length=256. Векторы нормализованы,
поиск — точное скалярное произведение.

In [4]:
weights = snapshot_download(MODEL_ID, revision=MODEL_REVISION, cache_dir=str(MODEL_CACHE),
    local_files_only=not ALLOW_MODEL_DOWNLOAD,
    allow_patterns=['*.json', '*.txt', 'sentencepiece.bpe.model', 'tokenizer.model', 'model.safetensors'])
model = SentenceTransformer(weights, device=DEVICE, local_files_only=True)
model.max_seq_length = 256
if DEVICE.startswith('cuda'):
    model.half()
model.eval()


def normalized_vectors(values):
    values = np.asarray(values, dtype=np.float32)
    return values / np.maximum(np.linalg.norm(values, axis=1, keepdims=True), 1e-12)

def item_vectors(corpus, model):
    path = CACHE/'item_vectors.npy'
    if path.exists():
        return np.load(path, mmap_mode='r')
    texts = ['passage: '+str(row.item_title_raw)+'\n'
        +' '.join(str(row.item_infm_params_text).split()[:64])+'\n'
        +' '.join(str(row.item_description_raw).split()[:160])
        for row in corpus[list(FIELDS)].fillna('').itertuples(index=False)]
    partial = path.with_suffix('.partial.npy')
    values = np.lib.format.open_memmap(partial, mode='w+', dtype=np.float32, shape=(len(corpus),384))
    # Размеры блоков и batch_size фиксированы: fp16 может зависеть от формы батча.
    for start in range(0,len(texts),8192):
        block = texts[start:start+8192]
        encoded = model.encode(block, batch_size=128, normalize_embeddings=True,
                               convert_to_numpy=True, show_progress_bar=False)
        values[start:start+len(block)] = normalized_vectors(encoded)
    values.flush()
    del values
    gc.collect()
    partial.replace(path)
    return np.load(path, mmap_mode='r')

vectors = item_vectors(corpus, model)
query_vectors = normalized_vectors(model.encode(['query: '+str(text) for text in queries.search_query],
    batch_size=128, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False))
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4230.51it/s]


## BM25 и контекст

Веса заголовка/параметров/описания — 3:2:1, описание ограничено 300 токенами.
Отдельный индекс — после Snowball. География, фильтры и порог рейтинга
учитываются мягкими бонусами без жёсткого отсечения.

In [5]:
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.deg2rad, (lat1, lon1, lat2, lon2))
    value = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 6371.0088 * 2 * np.arcsin(np.sqrt(np.clip(value, 0, 1)))

def location_centres(metadata):
    points = metadata[['item_id', 'item_location_id', 'item_latitude', 'item_longitude']].copy()
    for column in ['item_latitude', 'item_longitude']:
        points[column] = pd.to_numeric(points[column], errors='coerce').astype(float)
    valid = points.item_latitude.between(-90, 90) & points.item_longitude.between(-180, 180)
    valid &= ~((points.item_latitude == 0) & (points.item_longitude == 0)) & (points.item_location_id > 0)
    points = points[valid]
    centres = points.groupby('item_location_id').agg(
        lat=('item_latitude', 'median'), lon=('item_longitude', 'median'), count=('item_id', 'size'))
    points['spread'] = haversine(points.item_latitude.to_numpy(), points.item_longitude.to_numpy(),
        points.item_location_id.map(centres.lat).to_numpy(), points.item_location_id.map(centres.lon).to_numpy())
    centres['spread'] = points.groupby('item_location_id').spread.median()
    return centres[(centres['count'] >= 3) & (centres.spread <= 100)]

class CandidateRetriever:
    def __init__(self, corpus, metadata):
        self.corpus = corpus
        self.ids = corpus.item_id.astype(str).to_numpy(dtype='U16')
        self.locations = corpus.item_location_id.fillna(-1).to_numpy()
        self.ratings = pd.to_numeric(corpus.item_rating, errors='coerce').to_numpy(float)
        self.centres = location_centres(metadata)
        locations, self.location_inverse = np.unique(self.locations, return_inverse=True)
        centres = self.centres.reindex(locations)
        self.lat, self.lon = centres.lat.to_numpy(), centres.lon.to_numpy()
        # Переиспользуем совместимость одинаковых фильтров/локаций, не ответы на запросы.
        self.geo_cache, self.param_cache, self.rating_cache = {}, {}, {}
        self.indexes, self.stem_indexes = {}, {}
        for field in FIELDS:
            index_path = CACHE / f'{field}_bm25'
            stem_path = CACHE / f'{field}_snowball'
            if (index_path/'params.index.json').exists() and (stem_path/'params.index.json').exists():
                self.indexes[field] = bm25s.BM25.load(index_path, mmap=True, load_corpus=False)
                self.stem_indexes[field] = bm25s.BM25.load(stem_path, mmap=True, load_corpus=False)
                print('Индекс загружен:', field)
                continue
            docs = [tokenize(x) for x in corpus[field].fillna('')]
            if field == 'item_description_raw':
                docs = [d[:300] for d in docs]
            index = bm25s.BM25(k1=1.2, b=.75, method='lucene')
            index.index(docs, show_progress=False)
            self.indexes[field] = index
            index.save(index_path, show_progress=False)
            vocabulary = list(set(t for doc in docs for t in doc))
            stems = dict(zip(vocabulary, STEMMER.stemWords(vocabulary)))
            index = bm25s.BM25(k1=1.2, b=.75, method='lucene')
            index.index([[stems[t] for t in doc] for doc in docs], show_progress=False)
            self.stem_indexes[field] = index
            index.save(stem_path, show_progress=False)
            del docs, vocabulary, stems
            gc.collect()
            print('Индекс построен:', field)

    def context(self, row):
        text = str(row.search_infm_params_text or '').lower().replace('ё', 'е')
        match = RATING_PATTERN.search(text)
        rating = float(match.group(1).replace(',', '.')) if match else None
        params = sorted(set(tokenize(LABEL_PATTERN.sub(' ', RATING_PATTERN.sub(' ', text)))) - STOPWORDS)
        param_key = tuple(params)
        if param_key not in self.param_cache:
            score = np.zeros(len(self.ids), np.float32)
            if params:
                score = self.indexes['item_infm_params_text'].get_scores(params)
                if score.max() > 0:
                    score /= score.max()
            self.param_cache[param_key] = score
        if rating not in self.rating_cache:
            self.rating_cache[rating] = np.isfinite(self.ratings) & (self.ratings >= rating) if rating is not None else False
        use_location = row.search_is_delivery_search != 1 and pd.notna(row.search_location_id) and row.search_location_id > 0
        geo_key = int(row.search_location_id) if use_location else None
        if geo_key not in self.geo_cache:
            # Доставка отключает локальный бонус; иначе соседние города тоже допустимы.
            affinity = np.zeros(len(self.ids), np.float32)
            if use_location:
                affinity = (self.locations == row.search_location_id).astype(np.float32)
                if row.search_location_id in self.centres.index:
                    centre = self.centres.loc[row.search_location_id]
                    distance = haversine(centre.lat, centre.lon, self.lat, self.lon)
                    nearby = (.5 * np.exp(-np.nan_to_num(distance, nan=np.inf)/75)).astype(np.float32)
                    affinity = np.maximum(affinity, nearby[self.location_inverse])
            self.geo_cache[geo_key] = np.exp(np.log(5.)*affinity)
        return self.geo_cache[geo_key] * (1 + self.param_cache[param_key] + .15*self.rating_cache[rating])

engine = CandidateRetriever(corpus, geography_metadata(corpus))

Индекс загружен: item_title_raw
Индекс загружен: item_infm_params_text
Индекс загружен: item_description_raw


## RRF

Raw BM25 + E5: 35%/65%; затем морфология: 30%; расширенный BM25: 15%.
Вес добавленных термов — 0.3. Константа RRF — 30, глубина промежуточной
выдачи — 500. Малый вклад расширений ограничивает query drift.

In [6]:
def rrf(rankings, weights, k=DEPTH, constant=30):
    active = [(np.asarray(r)[np.asarray(r) >= 0], w) for r, w in zip(rankings, weights) if w > 0]
    active = [(r, w) for r, w in active if len(r)]
    if not active:
        return np.array([], dtype=np.int32)
    positions = np.concatenate([r for r, w in active])
    values = np.concatenate([w / (constant + np.arange(1, len(r)+1)) for r, w in active])
    unique, inverse = np.unique(positions, return_inverse=True)
    scores = np.zeros(len(unique), dtype=np.float64)
    np.add.at(scores, inverse, values)
    return unique[top(scores, k)]

def weighted_bm25(index,terms):
    """Weighted sum of the same BM25 postings; no dense term/document matrix."""
    scores = np.zeros(index.scores['num_docs'],np.float32)
    for term,weight in terms:
        token = index.vocab_dict.get(term)
        if token is None:
            continue
        start,end = index.scores['indptr'][token:token+2]
        scores[index.scores['indices'][start:end]] += weight*index.scores['data'][start:end]
    return scores

def retrieve_candidates(queries, query_vectors, engine, vectors, expansion):
    rows = list(queries[QUERY_COLUMNS].itertuples(index=False))
    tensor = torch.from_numpy(np.asarray(vectors).copy()).to(DEVICE)
    answers = []
    with torch.inference_mode():
        for start in range(0,len(rows),16):
            cosine = (torch.from_numpy(query_vectors[start:start+16]).to(DEVICE) @ tensor.T).cpu().numpy()
            for offset,row in enumerate(rows[start:start+16]):
                factor = engine.context(row)
                tokens = sorted(set(tokenize(row.search_query)))
                stems = sorted(set(STEMMER.stemWords(tokens)))
                raw,morph,extra = [np.zeros(len(engine.ids),np.float32) for _ in range(3)]
                expanded_terms = expansion.expand(row.search_query,limit=4)
                for field,weight in FIELDS.items():
                    if tokens:
                        raw += weight*engine.indexes[field].get_scores(tokens)
                        morph += weight*engine.stem_indexes[field].get_scores(stems)
                    extra += weight*weighted_bm25(engine.stem_indexes[field],expanded_terms)
                lexical = top(raw*factor,positive=True)
                morphology = top(morph*factor,positive=True)
                semantic = top(cosine[offset]+.05*np.log(factor))
                expanded = top((morph+.3*extra)*factor,positive=True)
                # Три ступени RRF, constant=30, по 500 кандидатов на промежуточных стадиях.
                hybrid = rrf([lexical,semantic],[.35,.65])
                current = rrf([hybrid,morphology],[.7,.3])
                answers.append(rrf([current,expanded],[.85,.15],k=50))
            if start%640==0:
                print('Поиск:',min(start+16,len(rows)),'/',len(rows))
    return answers

candidates = retrieve_candidates(queries,query_vectors,engine,vectors,expansion)

Поиск: 16 / 2452
Поиск: 656 / 2452
Поиск: 1296 / 2452
Поиск: 1936 / 2452


## Экспорт

In [7]:
answer = pd.DataFrame({'query_id':queries.query_id.astype(str),
    'answer':[' '.join(engine.ids[positions]) for positions in candidates]})
valid_ids = set(engine.ids)
assert len(answer)==len(queries) and answer.query_id.is_unique
for line in answer.answer:
    ids = line.split()
    assert len(ids)==len(set(ids))==50 and set(ids)<=valid_ids
    assert all(re.fullmatch('[0-9a-f]{16}',item) for item in ids)
answer.to_csv(ROOT/'answer.csv',index=False,encoding='utf-8',lineterminator='\n')
print('Готово:',len(answer),'запросов, по 50 кандидатов')

Готово: 2452 запросов, по 50 кандидатов


## Валидация

Разбиение — по нормализованным текстам запросов. Из обучения расширения
исключены проверочные тексты и их положительные объявления. Поиск проверялся
среди полного train + benchmark корпуса; параметры выбирались на dev,
отдельный control использовался только для подтверждения эффекта.

Основные ошибки — несовпадение словоформ и смешение соседних услуг при
расширении. Используются стемминг, ограничение числа и веса добавленных термов.